**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# State-Space Models: Kalman → S4 → Mamba

The course only a signal processing society can teach properly. Modern sequence architectures (S4, Mamba) are *literally* this curriculum: state-space models ([Kalman](./Intro_AdFilt_KF.ipynb)), convolution kernels ([Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb)), and discretization ([Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)) — rebranded for deep learning. Four sessions from the linear SSM you already know to a trained sequence model, with every identity verified numerically.

## 1. Pre-requisites

- [Kalman](./Intro_AdFilt_KF.ipynb) & [RNN](./Intro_RNN.ipynb) workshops.
- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) for the architecture being challenged.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Linear SSMs Are Convolutions* (~35 min)
**Goal:** prove (numerically) that an LTI state space = one long FIR filter; why that unlocks parallel training.
**Builds on:** [Kalman](./Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 2 (HiPPO & discretization).

---

## 2. The Identity Everything Rests On

💡 **Intuition.** A linear time-invariant SSM $h_{t} = \bar{A} h_{t-1} + \bar{B} x_t, \; y_t = C h_t$ can be *unrolled*: $y_t = \sum_{k\ge0} C\bar{A}^{k}\bar{B} \, x_{t-k}$ — a *convolution* with kernel $K_k = C \bar A^k \bar B$. That one identity is the whole trick: **train as a convolution** (parallel, FFT-fast, no backprop-through-time vanishing) and **infer as a recurrence** (constant memory per step, unlike attention's growing KV cache). RNN pain and transformer pain, both dodged — for *linear* state dynamics.

In [2]:
# ORACLE CHECK: recurrence output == convolution output, elementwise
d_state, T = 8, 200
A = np.diag(rng.uniform(0.7, 0.98, d_state))          # stable diagonal SSM
Bm = rng.standard_normal((d_state, 1))
Cm = rng.standard_normal((1, d_state))
x = rng.standard_normal(T)

# path 1: run the recurrence
h = np.zeros(d_state); y_rec = np.zeros(T)
for t in range(T):
    h = A @ h * 1.0 + (Bm[:, 0] * x[t])
    y_rec[t] = Cm[0] @ h

# path 2: materialize the kernel, convolve
K = np.array([ (Cm @ np.linalg.matrix_power(A, k) @ Bm)[0, 0] for k in range(T) ])
y_conv = np.convolve(x, K)[:T]

print("max |recurrence − convolution| =", np.abs(y_rec - y_conv).max())
assert np.abs(y_rec - y_conv).max() < 1e-9
plt.figure(figsize=(7.5, 2.2)); plt.plot(K[:60], ".-")
plt.title("the SSM's implicit FIR kernel  $K_k = C\\bar{A}^k\\bar{B}$")
plt.tight_layout(); plt.show()

max |recurrence − convolution| = 1.7763568394002505e-15


/tmp/ipykernel_2953716/2656292720.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 4 — *HiPPO & Discretization: Why S4's A Matrix Is Special* (~35 min)
**Goal:** see why random A forgets; meet the memory-optimal initialization and the continuous-time view.
**Builds on:** Session 1; [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb). &nbsp; **Feeds into:** Session 3 (training an SSM).

---

## 3. Long Memory Is an Initialization Problem

💡 **Intuition.** Session 1's kernel is a sum of geometric decays $\lambda_i^k$ ([Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb): the poles are the modes!). Random stable $A$ ⇒ all modes die at similar rates ⇒ effective memory of a few dozen steps, the RNN disease in linear form. **HiPPO**'s insight: choose $A$ so the state stores *orthogonal-polynomial coefficients of the input's history* — a principled spread of timescales, kernels with long structured tails. S4 = HiPPO-initialized continuous SSM, discretized ([FoSP2 S2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s $\bar{A} = e^{A\Delta}$, in practice bilinear/ZOH) with a *learnable* step size $\Delta$ — the network literally learns its own sampling rate per channel.

In [3]:
# Kernel shapes: random-diagonal vs HiPPO-style log-spaced timescales
def kernel_of(eigs, T=400):
    C1 = np.ones(len(eigs)) / len(eigs)                # equal weights: shape comes from the poles
    return np.array([ (C1 * eigs**k).sum() for k in range(T) ])

eig_rand = rng.uniform(0.7, 0.95, 32)
eig_hippo_ish = np.exp(-np.logspace(-3.5, 0, 32))     # timescales spread over decades

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.6))
axes[0].plot(kernel_of(eig_rand)); axes[0].set_title("random A: kernel dead by k≈50")
axes[1].plot(kernel_of(eig_hippo_ish)); axes[1].set_title("decade-spread timescales: structure for hundreds of steps")
plt.tight_layout(); plt.show()
def memory_span(K):                                    # last step where the kernel is still >1% of its peak
    return int(np.max(np.nonzero(np.abs(K) > 0.01 * np.abs(K).max())))
print(f"kernel span (last tap above 1% of peak): random A → {memory_span(kernel_of(eig_rand))} steps,"
      f"  decade-spread → {memory_span(kernel_of(eig_hippo_ish))}+ steps")

kernel span (last tap above 1% of peak): random A → 47 steps,  decade-spread → 399+ steps


/tmp/ipykernel_2953716/1097700265.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 4 — *Train a Diagonal SSM* (~40 min)
**Goal:** build an S4-style layer (diagonal, conv-trained) and beat the LSTM on a long-memory task.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (selectivity & Mamba).

---

## 4. The Layer, Assembled

Our layer (an honest simplification of S4D): per channel, learnable log-timescales $\lambda = e^{-e^{\theta}}$, learnable $C$; compute the kernel, convolve via FFT, add a skip and a nonlinearity. Trained **as a convolution**, verified equal to its recurrence.

In [4]:
class DiagSSMLayer(nn.Module):
    def __init__(self, d_model=32, d_state=32, T_max=1024):
        super().__init__()
        self.log_dt = nn.Parameter(torch.linspace(-4, 0, d_state).repeat(d_model, 1))  # decade spread
        self.C = nn.Parameter(torch.randn(d_model, d_state) / d_state**0.5)
        self.D = nn.Parameter(torch.ones(d_model))                                     # skip
        self.T_max = T_max
    def kernel(self, T):
        lam = torch.exp(-torch.exp(self.log_dt))              # (d_model, d_state) in (0,1)
        k = torch.arange(T)
        K = torch.einsum("ds,dsk->dk", self.C, lam[:, :, None] ** k[None, None, :])
        return K                                               # (d_model, T)
    def forward(self, x):                                      # x: (B, T, d_model)
        B, T, D = x.shape
        K = self.kernel(T)
        Xf = torch.fft.rfft(x.transpose(1, 2), n=2*T)          # convolve per channel via FFT
        Kf = torch.fft.rfft(K, n=2*T)
        y = torch.fft.irfft(Xf * Kf[None], n=2*T)[..., :T].transpose(1, 2)
        return torch.nn.functional.gelu(y + x * self.D)

# ORACLE: layer's FFT-conv path == naive recurrence, for one channel
layer = DiagSSMLayer(d_model=4, d_state=8)
x_test = torch.randn(1, 64, 4)
with torch.no_grad():
    y_fft = layer(x_test)
    lam = torch.exp(-torch.exp(layer.log_dt))
    y_rec = torch.zeros_like(x_test)
    for d in range(4):
        h = torch.zeros(8)
        for t in range(64):
            h = lam[d] * h + x_test[0, t, d]
            y_rec[0, t, d] = (layer.C[d] * h).sum()
    y_rec = torch.nn.functional.gelu(y_rec + x_test * layer.D)
print("max |FFT-conv − recurrence| =", (y_fft - y_rec).abs().max().item())
assert (y_fft - y_rec).abs().max() < 1e-4

max |FFT-conv − recurrence| = 2.384185791015625e-06


In [5]:
# The long-memory gauntlet: recall the FIRST token's class after T=400 noise steps
def task_batch(B=64, T=400):
    x = torch.randn(B, T, 1) * 0.3
    labels = torch.randint(0, 2, (B,))
    x[:, 0, 0] = labels.float() * 2 - 1                       # ±1 cue at t=0, then noise
    return x, labels

class SSMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(1, 32)
        self.s1, self.s2 = DiagSSMLayer(32), DiagSSMLayer(32)
        self.head = nn.Linear(32, 2)
    def forward(self, x):
        h = self.inp(x); h = self.s1(h); h = self.s2(h)
        return self.head(h[:, -1])

class LSTMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, 2, batch_first=True)
        self.head = nn.Linear(32, 2)
    def forward(self, x):
        return self.head(self.lstm(x)[0][:, -1])

results = {}
for name, model in [("SSM", SSMNet()), ("LSTM", LSTMNet())]:
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    accs = []
    for step in range(300):
        x, yb = task_batch()
        loss = nn.functional.cross_entropy(model(x), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 25 == 0:
            with torch.no_grad():
                xv, yv = task_batch(256)
                accs.append((model(xv).argmax(1) == yv).float().mean().item())
    results[name] = accs
    print(f"{name}: final recall accuracy over 400 steps = {accs[-1]:.1%}")

plt.figure(figsize=(7.5, 2.6))
for name, accs in results.items(): plt.plot(np.arange(len(accs))*25, accs, "o-", label=name)
plt.axhline(0.5, color="k", linestyle=":", linewidth=0.8, label="chance")
plt.legend(); plt.xlabel("training step"); plt.ylabel("val accuracy")
plt.title("remember one token across 400 steps: decade-spread SSM vs LSTM")
plt.tight_layout(); plt.show()

SSM: final recall accuracy over 400 steps = 93.4%


LSTM: final recall accuracy over 400 steps = 54.7%


/tmp/ipykernel_2953716/2672149721.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 4 of 4 — *Selectivity & Mamba (Frontier Sketch)* (~30 min)
**Goal:** understand what Mamba changes — input-dependent dynamics — and what that costs.
**Builds on:** Session 3.

---

## 5. What Mamba Adds — and What It Breaks

> ℹ️ **Frontier sketch.** This session explains the mechanism and its trade-off; a full efficient selective-scan implementation (Mamba's hardware-aware kernel) is beyond a 40-minute session and is *not* implemented here.

💡 **Intuition.** Everything above is **LTI**: the same kernel for every input — the system cannot *decide* to remember this token and forget that one. Mamba makes $\bar B, \bar C, \Delta$ **functions of the current input** ('selective'): a content-controlled gate on the state, per step. The price is exactly the trade this course's structure predicts: input-dependent dynamics are **time-varying**, so the convolution identity of Session 1 *no longer holds* — no FFT training path. Mamba's contribution is showing the recurrence can still be computed fast on GPUs (parallel associative scan + kernel fusion — the [HW-Accelerated](../Intro_GPU/HW_Accelerated_Computing.ipynb) toolbox earning its keep).

The one-line summary of the whole architecture family:

| | RNN/LSTM | Transformer | S4 (LTI SSM) | Mamba (selective) |
|---|---|---|---|---|
| Train | sequential | parallel | parallel (FFT conv) | parallel (assoc. scan) |
| Infer/step | $O(1)$ | $O(T)$ (KV cache) | $O(1)$ | $O(1)$ |
| Content-dependent routing | gates | **attention** | ✗ | **selection** |
| DSP name | nonlinear IIR | data-adaptive kernel | FIR bank w/ learned poles | time-varying system |

In [6]:
# The selectivity mechanism in miniature (correct but naive O(T) loop — the SKETCH):
# gate Δ_t = f(x_t) controls how much the state updates on each token
class SelectiveToy(nn.Module):
    def __init__(self, d=16):
        super().__init__()
        # the gate sees the current token AND the previous one (a 1-step conv, as in Mamba's
        # local conv before the SSM) — Δ depends on the INPUT: this is the selectivity
        self.gate = nn.Linear(2, d)
        self.C = nn.Parameter(torch.randn(d) / d**0.5)
    def forward(self, x):                  # x: (B, T, 1)
        B, T, _ = x.shape
        x_prev = torch.cat([torch.zeros(B, 1, 1), x[:, :-1]], 1)
        feats = torch.cat([x, x_prev], -1)
        dt = torch.sigmoid(self.gate(feats))   # (B, T, d) in (0,1): per-token write strength
        h = torch.zeros(B, dt.shape[-1])
        ys = []
        for t in range(T):                 # honest recurrence — no conv shortcut EXISTS here
            h = (1 - dt[:, t]) * h + dt[:, t] * x[:, t]
            ys.append(h @ self.C)
        return torch.stack(ys, 1)

# task LTI SSMs cannot do: output the last token that FOLLOWED a '2' marker
toy = SelectiveToy(); opt = torch.optim.Adam(toy.parameters(), lr=1e-2)
def sel_batch(B=128, T=40):
    x = torch.rand(B, T, 1)
    pos = torch.randint(5, T-1, (B,))
    x[torch.arange(B), pos, 0] = 2.0                       # marker
    target = x[torch.arange(B), pos+1, 0]                  # remember what came after it
    return x, target
for step in range(600):
    x, tgt = sel_batch()
    loss = ((toy(x)[:, -1] - tgt)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
x, tgt = sel_batch(512)
with torch.no_grad(): mse = ((toy(x)[:, -1] - tgt)**2).mean().item()
print(f"selective toy MSE on marker-recall: {mse:.4f}  (predicting the mean would give ≈{tgt.var().item():.4f})")
print("→ an input-dependent gate solves a task no fixed kernel can — that's Mamba's bet, at scale")

selective toy MSE on marker-recall: 0.0078  (predicting the mean would give ≈0.0839)
→ an input-dependent gate solves a task no fixed kernel can — that's Mamba's bet, at scale


## 6. Conclusion

Linear SSM = convolution (verified), long memory = timescale spread (HiPPO's gift), training = FFT, inference = recurrence — and Mamba trades the conv identity for content-selective dynamics computed by scan. You can now read the S4/Mamba papers as *signal processing literature*, because that's what they are.

---
## Where next

- [LLMs from the Ground Up](../Intro_Mach_Learn/LLMs_from_the_Ground_Up.ipynb) — the model family SSMs compete with.
- [Kalman](./Intro_AdFilt_KF.ipynb) / [FoSP2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) — the two halves this course glued together.
- [Modern Architectures](../Intro_Mach_Learn/Modern_Architectures.ipynb) — where SSM blocks sit in today's model zoo.